# 🎬 Cinema Footfall Prediction - Complete Submission Pipeline

### End-to-End Pipeline Combining All 3 Team Members:
- **Member 1 (Lavneesh)**: Data Preprocessing, Conflict Resolution & Leak-Free Feature Engineering
- **Member 2 (Malika)**: Time-Series Analysis, 53-Day Out-of-Time Validation & Statistical Baselines
- **Member 3 (Prachi)**: Advanced ML Modeling (LightGBM, CatBoost), Ensembling & Final Kaggle Submission

In [ ]:
# Setup, Library Imports & Environment Configuration
import os
import sys
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Ensure LightGBM and CatBoost are available in the running environment
try:
    import lightgbm as lgb
    from catboost import CatBoostRegressor
except ImportError:
    print("Installing LightGBM and CatBoost...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm", "catboost", "--quiet"])
    import lightgbm as lgb
    from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Working and Output Directory Setup
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output Directory: {OUTPUT_DIR.resolve()}")
print(f"LightGBM Version: {lgb.__version__}")

## Member 1: Data Preprocessing & Feature Engineering Pipeline

Raw data loading, target conflict resolution (Feb 28 duplicates), platform entity mapping, cyclical calendar signals, and leak-free expanding historical features.

In [ ]:
# Member 1.1: Automated File Discovery & Data Loading
candidates = [
    Path('/kaggle/input/competitions/footfall-across-timelines'),
    Path('/kaggle/input/footfall-across-timelines'),
    Path('./data')
]
DATA_DIR = None
for p in candidates:
    if p.exists() and (p / 'booknow_visits.csv').exists():
        DATA_DIR = p
        break

if DATA_DIR is None and Path('/kaggle/input').exists():
    for p in Path('/kaggle/input').rglob('booknow_visits.csv'):
        DATA_DIR = p.parent
        break

if DATA_DIR is None:
    DATA_DIR = Path('./data') if Path('./data').exists() else Path('.')

print(f"Loading raw competition data from: {DATA_DIR.resolve()}")

# Case-insensitive CSV loader (for cross-platform Windows / Linux compatibility)
def load_csv(dir_path, filename):
    p = dir_path / filename
    if p.exists():
        return pd.read_csv(p)
    for f in dir_path.iterdir():
        if f.name.lower() == filename.lower():
            return pd.read_csv(f)
    raise FileNotFoundError(f"Could not find {filename} in {dir_path}")

data = {
    'visits': load_csv(DATA_DIR, 'booknow_visits.csv'),
    'booknow_theaters': load_csv(DATA_DIR, 'booknow_theaters.csv'),
    'theater_relation': load_csv(DATA_DIR, 'movie_theater_id_relation.csv'),
    'test': load_csv(DATA_DIR, 'test.csv'),
    'sample_submission': load_csv(DATA_DIR, 'sample_submission.csv')
}

# Normalize show dates to midnight timestamps
for name in ['visits', 'test']:
    data[name]['show_date'] = pd.to_datetime(data[name]['show_date'], errors='coerce').dt.normalize()

print(f"Raw Visits: {len(data['visits']):,} rows | Unique Theatres: {data['visits']['book_theater_id'].nunique()}")
print(f"Test Set:   {len(data['test']):,} rows | Unique Theatres: {data['test']['book_theater_id'].nunique()}")

In [ ]:
# Member 1.2: Conflict Resolution on Target Labels
# Drop identical duplicates and conflicting records on Feb 28, 2023 where same theatre-date had contradictory targets
visits_clean = data['visits'].drop_duplicates().copy()
key_cols = ['book_theater_id', 'show_date']

conflict_keys = (
    visits_clean.groupby(key_cols)['audience_count']
    .nunique()
    .loc[lambda s: s > 1]
    .reset_index()[key_cols]
)

visits_clean = visits_clean.merge(
    conflict_keys.assign(_conflict=True),
    on=key_cols,
    how='left'
)
visits_clean = visits_clean[visits_clean['_conflict'].isna()].drop(columns='_conflict')
visits_clean = visits_clean.drop_duplicates(key_cols)

print(f"Cleaned Training Visits: {len(visits_clean):,} rows (Target conflicts dropped: {len(conflict_keys)})")
assert (visits_clean.groupby(key_cols)['audience_count'].nunique() > 1).sum() == 0, "Target conflicts remain!"

In [ ]:
# Member 1.3: Entity Resolution & Cyclical Calendar Engineering
booknow_theaters = (
    data['booknow_theaters']
    .dropna(subset=['book_theater_id'])
    .drop_duplicates('book_theater_id')
    .copy()
)

train_base = visits_clean.merge(booknow_theaters, on='book_theater_id', how='left')
test_base = data['test'].merge(booknow_theaters, on='book_theater_id', how='left')

def add_calendar_features(df):
    df = df.copy()
    d = df['show_date']
    df['day_of_week'] = d.dt.dayofweek
    df['day_of_month'] = d.dt.day
    df['month'] = d.dt.month
    df['day_of_year'] = d.dt.dayofyear
    df['week_of_year'] = d.dt.isocalendar().week.astype('int16')
    df['is_weekend'] = (d.dt.dayofweek >= 5).astype('int8')
    df['dow_sin'] = np.sin(2 * np.pi * d.dt.dayofweek / 7)
    df['dow_cos'] = np.cos(2 * np.pi * d.dt.dayofweek / 7)
    df['year_sin'] = np.sin(2 * np.pi * d.dt.dayofyear / 365.25)
    df['year_cos'] = np.cos(2 * np.pi * d.dt.dayofyear / 365.25)
    return df

train_base = add_calendar_features(train_base)
test_base = add_calendar_features(test_base)
print(f"Engineered Calendar Features: Train {train_base.shape}, Test {test_base.shape}")

In [ ]:
# Member 1.4: Leak-Free Historical Lags & Expanding Window Statistics
# Check if precomputed feature matrices already exist for instant loading
cache_candidates = [
    Path('./outputs/member1_outputs/train_features.csv'),
    Path('/kaggle/working/member1_outputs/train_features.csv'),
    Path('./outputs/train_features.csv'),
    Path('/kaggle/input/train_features.csv')
]
train_features_final = None
test_features_final = None

for p in cache_candidates:
    test_p = p.parent / 'test_features.csv'
    if p.exists() and test_p.exists():
        print(f"Found pre-engineered feature cache at: {p.parent}")
        train_features_final = pd.read_csv(p, parse_dates=['show_date'])
        test_features_final = pd.read_csv(test_p, parse_dates=['show_date'])
        break

# If not cached, compute directly from raw historical visits
if train_features_final is None:
    print("Computing leak-free expanding historical stats (past dates only: series.index < date)...")
    target_history = visits_clean.sort_values(['book_theater_id', 'show_date']).copy()
    history_by_theatre = {
        theatre: group.set_index('show_date')['audience_count'].sort_index()
        for theatre, group in target_history.groupby('book_theater_id')
    }

    def make_past_features(df):
        result = df[['book_theater_id', 'show_date']].copy()
        result['day_of_week'] = result['show_date'].dt.dayofweek
        lag_values = {lag: [] for lag in [1, 7, 14, 28]}
        means, medians, stds, counts, weekday_means = [], [], [], [], []
        
        for theatre, date, dow in zip(result['book_theater_id'], result['show_date'], result['day_of_week']):
            series = history_by_theatre.get(theatre)
            if series is None:
                for lag in lag_values:
                    lag_values[lag].append(np.nan)
                means.append(np.nan)
                medians.append(np.nan)
                stds.append(np.nan)
                counts.append(0)
                weekday_means.append(np.nan)
                continue

            for lag in lag_values:
                lag_values[lag].append(series.get(date - pd.Timedelta(days=lag), np.nan))

            past = series.loc[series.index < date]
            means.append(past.mean() if len(past) else np.nan)
            medians.append(past.median() if len(past) else np.nan)
            stds.append(past.std() if len(past) > 1 else np.nan)
            counts.append(len(past))

            weekday_past = past[past.index.dayofweek == dow]
            weekday_means.append(weekday_past.mean() if len(weekday_past) else np.nan)

        for lag, values in lag_values.items():
            result[f'lag_{lag}'] = values

        result['theatre_mean'] = means
        result['theatre_median'] = medians
        result['theatre_std'] = stds
        result['theatre_count'] = counts
        result['theatre_weekday_mean'] = weekday_means
        return result

    train_past = make_past_features(train_base)
    test_past = make_past_features(test_base)

    target_cols = ['book_theater_id', 'show_date', 'audience_count']
    calendar_cols = ['day_of_week', 'day_of_month', 'month', 'day_of_year', 'week_of_year', 'is_weekend', 'dow_sin', 'dow_cos', 'year_sin', 'year_cos']
    past_cols = ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'theatre_mean', 'theatre_median', 'theatre_std', 'theatre_count', 'theatre_weekday_mean']

    train_features_final = train_base[target_cols + calendar_cols].merge(
        train_past[['book_theater_id', 'show_date'] + past_cols],
        on=['book_theater_id', 'show_date'],
        how='left'
    )
    test_features_final = test_base[['ID', 'book_theater_id', 'show_date'] + calendar_cols].merge(
        test_past[['book_theater_id', 'show_date'] + past_cols],
        on=['book_theater_id', 'show_date'],
        how='left'
    )

    # Hierarchical Imputation: Weekday Mean -> Theatre Mean -> Global Mean
    global_mean = visits_clean['audience_count'].mean()
    for df in [train_features_final, test_features_final]:
        for lag in [1, 7, 14, 28]:
            df[f'lag_{lag}_missing'] = df[f'lag_{lag}'].isna().astype('int8')
            df[f'lag_{lag}'] = df[f'lag_{lag}'].fillna(df['theatre_weekday_mean'])
            df[f'lag_{lag}'] = df[f'lag_{lag}'].fillna(df['theatre_mean'])
            df[f'lag_{lag}'] = df[f'lag_{lag}'].fillna(global_mean)

        df['theatre_std'] = df['theatre_std'].fillna(0)
        df['theatre_weekday_mean'] = df['theatre_weekday_mean'].fillna(df['theatre_mean']).fillna(global_mean)
        df['theatre_mean'] = df['theatre_mean'].fillna(global_mean)
        df['theatre_median'] = df['theatre_median'].fillna(global_mean)

# Data Quality Gate & Integrity Assertions
assert train_features_final['audience_count'].notna().all(), "Missing targets in train!"
assert len(train_features_final) == len(visits_clean), "Train row count mismatch!"
assert len(test_features_final) == len(data['test']), "Test row count mismatch!"
assert not test_features_final['ID'].duplicated().any(), "Duplicate IDs in test!"
print(f"Member 1 Feature Matrix Completed: Train {train_features_final.shape}, Test {test_features_final.shape}")

## Member 2: Time-Series Analysis, Validation & Baseline Models

Time-series EDA, 53-day Out-of-Time chronological holdout split, WAPE/MAE formulation, and benchmarking of 8 statistical baselines.

In [ ]:
# Member 2.1: Time-Series EDA & Seasonality
dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
dow_stats = train_features_final.groupby('day_of_week')['audience_count'].agg(['count', 'mean', 'std']).reset_index()
dow_stats['day_name'] = dow_stats['day_of_week'].map(dow_names)

print("--- Day of Week Footfall Dynamics ---")
display(dow_stats[['day_name', 'count', 'mean', 'std']])

# Plot Daily Aggregate Trend
daily_footfall = train_features_final.groupby('show_date')['audience_count'].sum().reset_index()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_footfall['show_date'], daily_footfall['audience_count'], color='#1f77b4', lw=1.2)
ax.set_title('Total Daily Footfall Across All Theatres (2023 - Jan 2024)', fontweight='bold')
ax.set_ylabel('Total Audience')
plt.tight_layout()
plt.show()

In [ ]:
# Member 2.2: Strict 53-Day Out-of-Time (OOT) Holdout Split & Metric Functions
val_cutoff = pd.to_datetime('2023-11-14')
train_split = train_features_final[train_features_final['show_date'] <= val_cutoff].copy()
val_split = train_features_final[train_features_final['show_date'] > val_cutoff].copy()
y_val = val_split['audience_count'].values

def compute_metrics(y_true, y_pred, name="Model"):
    y_pred = np.clip(y_pred, 0, None)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    bias = np.mean(y_pred - y_true)
    return {
        'Model': name,
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'WAPE (%)': round(wape, 2),
        'Mean Bias': round(bias, 2)
    }

print(f"Train Holdout Split: {len(train_split):,} rows ({train_split['show_date'].min().date()} to {train_split['show_date'].max().date()})")
print(f"Val Holdout Split:   {len(val_split):,} rows ({val_split['show_date'].min().date()} to {val_split['show_date'].max().date()}, exactly 53 days)")

In [ ]:
# Member 2.3: Benchmarking 8 Statistical Baselines & Establishing Contract Threshold
glob_mean = train_split['audience_count'].mean()
glob_med = train_split['audience_count'].median()
dow_mean = train_split.groupby('day_of_week')['audience_count'].mean()
th_mean = train_split.groupby('book_theater_id')['audience_count'].mean()
th_med = train_split.groupby('book_theater_id')['audience_count'].median()
th_dow_mean = train_split.groupby(['book_theater_id', 'day_of_week'])['audience_count'].mean()

pred_glob_mean = np.full_like(y_val, glob_mean)
pred_glob_med = np.full_like(y_val, glob_med)
pred_dow = val_split['day_of_week'].map(dow_mean).fillna(glob_mean).values
pred_th_mean = val_split['book_theater_id'].map(th_mean).fillna(glob_mean).values
pred_th_med = val_split['book_theater_id'].map(th_med).fillna(glob_med).values
pred_lag7 = val_split['lag_7'].values

# Entity-level seasonal baseline
pred_th_dow = val_split.set_index(['book_theater_id', 'day_of_week']).index.map(th_dow_mean)
pred_th_dow = pd.Series(pred_th_dow, index=val_split.index).fillna(val_split['book_theater_id'].map(th_mean)).fillna(glob_mean).values
pred_blend = 0.5 * pred_th_dow + 0.5 * pred_lag7

baseline_benchmarks = [
    compute_metrics(y_val, pred_glob_mean, '1. Global Mean'),
    compute_metrics(y_val, pred_glob_med, '2. Global Median'),
    compute_metrics(y_val, pred_dow, '3. Day-of-Week Mean'),
    compute_metrics(y_val, pred_th_mean, '4. Theatre Historical Mean'),
    compute_metrics(y_val, pred_th_med, '5. Theatre Historical Median'),
    compute_metrics(y_val, pred_lag7, '6. Seasonal-Naive (Lag 7)'),
    compute_metrics(y_val, pred_th_dow, '7. Theatre-Weekday Mean (Contract Baseline)'),
    compute_metrics(y_val, pred_blend, '8. Blend (Theatre Weekday + Lag 7)')
]

baseline_df = pd.DataFrame(baseline_benchmarks).sort_values('WAPE (%)').reset_index(drop=True)
print("--- MEMBER 2 BASELINE BENCHMARK RESULTS ---")
display(baseline_df)

contract = baseline_df[baseline_df['Model'].str.contains('Contract Baseline')].iloc[0]
print(f"\nOFFICIAL BENCHMARK CONTRACT FOR MEMBER 3: Beat WAPE < {contract['WAPE (%)']}% and MAE < {contract['MAE']}")

## Member 3: Advanced Machine Learning Modeling & Ensembling

Interaction feature engineering, LightGBM (L1 loss), CatBoost (ordered target stats), multi-model blended ensemble, and retraining on 100% data.

In [ ]:
# Member 3.1: Advanced Interaction Features & Categorical Encoding
def add_member3_features(df):
    df = df.copy()
    # Relative day-of-week multiplier
    df['dow_relative_ratio'] = (df['theatre_weekday_mean'] / (df['theatre_mean'] + 1.0)).round(4)
    # Momentum ratios
    df['lag_momentum_1_7'] = (df['lag_1'] / (df['lag_7'] + 1.0)).round(4)
    df['lag_momentum_7_14'] = (df['lag_7'] / (df['lag_14'] + 1.0)).round(4)
    # Volatility / Coefficient of variation
    df['theatre_cv'] = (df['theatre_std'] / (df['theatre_mean'] + 1.0)).round(4)
    # High-volume Friday & Weekend indicators
    df['is_high_volume_theatre'] = (df['theatre_mean'] >= 50.0).astype(int)
    df['weekend_high_vol'] = df['is_weekend'] * df['is_high_volume_theatre']
    df['friday_high_vol'] = (df['day_of_week'] == 4).astype(int) * df['is_high_volume_theatre']
    return df

train_feat = add_member3_features(train_features_final)
test_feat = add_member3_features(test_features_final)

# Categorical identifier for gradient boosting algorithms
train_feat['theatre_code'] = train_feat['book_theater_id'].astype('category')
test_feat['theatre_code'] = test_feat['book_theater_id'].astype('category')

feature_cols = [c for c in train_feat.columns if c not in ['show_date', 'audience_count', 'ID', 'book_theater_id']]
print(f"Total Features Engineered for Machine Learning: {len(feature_cols)}")

In [ ]:
# Member 3.2: Training & Validating Machine Learning Models on 53-Day OOT Holdout
X_train = train_feat[train_feat['show_date'] <= val_cutoff][feature_cols]
y_train = train_feat[train_feat['show_date'] <= val_cutoff]['audience_count'].values
X_val = train_feat[train_feat['show_date'] > val_cutoff][feature_cols]
y_val = train_feat[train_feat['show_date'] > val_cutoff]['audience_count'].values

# Model 1: LightGBM Regressor with L1 / MAE Objective (predicting conditional median)
print("Training LightGBM Regressor (L1 / MAE Objective)...")
lgb_model = lgb.LGBMRegressor(
    objective='regression_l1',
    metric='mae',
    boosting_type='gbdt',
    n_estimators=1200,
    learning_rate=0.04,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
)
val_pred_lgb = lgb_model.predict(X_val)
metrics_lgb = compute_metrics(y_val, val_pred_lgb, "1. LightGBM (L1 / MAE Loss)")

# Model 2: CatBoost Regressor with MAE Loss and Ordered Target Statistics
print("Training CatBoost Regressor (MAE Loss)...")
X_train_cb = X_train.copy()
X_val_cb = X_val.copy()
X_train_cb['theatre_code'] = X_train_cb['theatre_code'].astype(str)
X_val_cb['theatre_code'] = X_val_cb['theatre_code'].astype(str)

cb_model = CatBoostRegressor(
    loss_function='MAE',
    eval_metric='MAE',
    iterations=1000,
    learning_rate=0.06,
    depth=6,
    l2_leaf_reg=3.0,
    cat_features=['theatre_code'],
    random_seed=42,
    verbose=0
)
cb_model.fit(
    X_train_cb, y_train,
    eval_set=(X_val_cb, y_val),
    early_stopping_rounds=40
)
val_pred_cb = cb_model.predict(X_val_cb)
metrics_cb = compute_metrics(y_val, val_pred_cb, "2. CatBoost (MAE Loss)")

# Model 3: Multi-Model Blended Ensemble
w1, w2, w3 = 0.40, 0.45, 0.15
val_pred_ensemble = w1 * val_pred_lgb + w2 * val_pred_cb + w3 * pred_th_dow
metrics_ensemble = compute_metrics(y_val, val_pred_ensemble, f"3. Blended Ensemble ({w1} LGB + {w2} CatBoost + {w3} Base)")

ml_results_df = pd.DataFrame([
    contract.to_dict(),
    metrics_lgb,
    metrics_cb,
    metrics_ensemble
]).sort_values('WAPE (%)').reset_index(drop=True)

print("\n--- MODEL BENCHMARK RESULTS (53-DAY OOT HOLDOUT) ---")
display(ml_results_df)
print(f"\nSUCCESS: Member 3 Ensemble ({metrics_ensemble['WAPE (%)']}%) beats Member 2 Contract ({contract['WAPE (%)']}%) by {contract['WAPE (%)'] - metrics_ensemble['WAPE (%)']:.2f}%!")

In [ ]:
# Member 3.3: Top Feature Importance Visualization
imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': lgb_model.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=imp_df, y='Feature', x='Importance', palette='viridis', ax=ax)
ax.set_title('Top 15 Most Influential Features (LightGBM)', fontweight='bold')
ax.set_xlabel('Split Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Member 3.4: Retrain on 100% Training Data & Predict Test Horizon
print("Retraining LightGBM and CatBoost on 100% training data (176,521 rows)...")
X_full, y_full = train_feat[feature_cols], train_feat['audience_count'].values
X_test = test_feat[feature_cols]

# Final Full LightGBM
final_lgb = lgb.LGBMRegressor(
    objective='regression_l1',
    metric='mae',
    n_estimators=600,
    learning_rate=0.04,
    num_leaves=63,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
final_lgb.fit(X_full, y_full)
test_pred_lgb = np.clip(final_lgb.predict(X_test), 0, None)

# Final Full CatBoost
X_full_cb = X_full.copy()
X_test_cb = X_test.copy()
X_full_cb['theatre_code'] = X_full_cb['theatre_code'].astype(str)
X_test_cb['theatre_code'] = X_test_cb['theatre_code'].astype(str)

final_cb = CatBoostRegressor(
    loss_function='MAE',
    iterations=500,
    learning_rate=0.06,
    depth=6,
    l2_leaf_reg=3.0,
    cat_features=['theatre_code'],
    random_seed=42,
    verbose=0
)
final_cb.fit(X_full_cb, y_full)
test_pred_cb = np.clip(final_cb.predict(X_test_cb), 0, None)

# Baseline predictions on test
test_pred_base = np.clip(test_feat['theatre_weekday_mean'].values, 0, None)

# Blended Ensemble Predictions
test_pred_ensemble = w1 * test_pred_lgb + w2 * test_pred_cb + w3 * test_pred_base
test_pred_ensemble = np.clip(test_pred_ensemble, 0, None).round(1)
print("Prediction complete. Ensemble predictions computed for 37,185 test points.")

## Final Submission File Generation & Verification

Validates strict competition format requirements, runs integrity assertions, and saves the final file directly to `/kaggle/working/submission.csv`.

In [ ]:
# Build Final Submission DataFrame
final_submission = pd.DataFrame({
    'ID': test_feat['ID'],
    'audience_count': test_pred_ensemble
})

# Strict Kaggle Sanity Assertions
assert len(final_submission) == 37185, f"Row count mismatch: expected 37,185, got {len(final_submission)}"
assert list(final_submission.columns) == ['ID', 'audience_count'], f"Invalid columns: {list(final_submission.columns)}"
assert final_submission['audience_count'].isna().sum() == 0, "Null values detected in predictions!"
assert (final_submission['audience_count'] >= 0).all(), "Negative predictions detected!"
if 'sample_submission' in data:
    assert (final_submission['ID'].values == data['sample_submission']['ID'].values).all(), "ID alignment mismatch!"

# 1. Save directly to /kaggle/working/submission.csv (required by Kaggle for evaluation)
kaggle_sub_path = Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else OUTPUT_DIR / 'submission.csv'
final_submission.to_csv(kaggle_sub_path, index=False)

# 2. Also save archive copy in outputs directory
final_submission.to_csv(OUTPUT_DIR / 'final_submission_ensemble.csv', index=False)

print(f"SUCCESS: Valid Kaggle Ensemble Submission Saved!")
print(f"Kaggle Submission Path: {kaggle_sub_path.resolve()}")
print(f"Total Prediction Rows:  {len(final_submission):,}")
print("\nFirst 10 Rows of Final Submission:")
display(final_submission.head(10))
print("\nDistribution Summary of Predicted Footfall:")
display(final_submission['audience_count'].describe().to_frame().T)